# Fase 4: Árbol de Decisiones - Modelado y Evaluación (CRISP-ML)

En esta fase entrenaremos y evaluaremos un modelo de **Árbol de Decisiones** para predecir el Puntaje de Adicción a Redes Sociales (`Addicted_Score`). El Árbol de Decisiones ofrece una representación visual e interpretable de las reglas de decisión aprendidas, complementando los resultados de la Regresión Lineal.

### Objetivos:
1. Cargar el dataset preprocesado por la fase ETL.
2. Preparar las variables predictoras y la variable objetivo.
3. Dividir el dataset en conjuntos de entrenamiento (80%) y prueba (20%).
4. Entrenar un `DecisionTreeRegressor` y ajustar su profundidad máxima.
5. Evaluar métricas de desempeño: $R^2$, MSE, RMSE y MAE.
6. Visualizar el árbol de decisión entrenado y la importancia de cada variable.
7. Comparar el rendimiento con el modelo de Regresión Lineal (Notebook 03).
8. Exportar el mejor modelo como `modelo_arbol.pkl`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

print('Librerías para Árbol de Decisiones cargadas exitosamente.')

## 1. Carga de Datos y Selección de Variables

In [ ]:
# Cargar el dataset limpio generado en 01_ETL.ipynb
df = pd.read_csv('Students_Social_Media_Addiction_Cleaned.csv')
print(f'Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas.')
df.head()

In [ ]:
# Definición de variables predictoras y variable objetivo
features = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night',
            'Mental_Health_Score', 'Conflicts_Over_Social_Media']
X = df[features]
y = df['Addicted_Score']

print(f'Variables predictoras (X): {features}')
print(f'Variable objetivo (y): Addicted_Score | Valores únicos: {sorted(y.unique())}')

## 2. División del Dataset (Entrenamiento 80% / Prueba 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Set de Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Set de Prueba      : {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)')

## 3. Entrenamiento del Árbol de Decisiones

Primero entrenamos el modelo sin restricciones de profundidad para entender su capacidad máxima, y luego aplicamos poda (ajuste de `max_depth`) para obtener un modelo más generalizable.

In [ ]:
# --- Modelo sin restricción de profundidad (árbol completo) ---
dt_full = DecisionTreeRegressor(random_state=42)
dt_full.fit(X_train, y_train)

y_pred_full = dt_full.predict(X_test)
r2_full = r2_score(y_test, y_pred_full)
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred_full))

print(f'Árbol Sin Poda   => R²: {r2_full:.4f} | RMSE: {rmse_full:.4f} | Profundidad: {dt_full.get_depth()}')

In [ ]:
# --- Búsqueda de la profundidad óptima mediante validación cruzada ---
depths = range(1, 16)
cv_scores = []

for d in depths:
    dt_cv = DecisionTreeRegressor(max_depth=d, random_state=42)
    scores = cross_val_score(dt_cv, X_train, y_train, cv=5, scoring='r2')
    cv_scores.append(scores.mean())

best_depth = depths[np.argmax(cv_scores)]
print(f'Mejor profundidad encontrada (CV 5 folds): {best_depth}')

# Graficar R² promedio de validación cruzada vs profundidad del árbol
plt.figure(figsize=(10, 5))
plt.plot(depths, cv_scores, marker='o', color='royalblue', linewidth=2, markersize=8)
plt.axvline(x=best_depth, color='crimson', linestyle='--', label=f'Profundidad óptima = {best_depth}')
plt.title('R² de Validación Cruzada vs. Profundidad del Árbol')
plt.xlabel('Profundidad Máxima (max_depth)')
plt.ylabel('R² Promedio (CV=5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- Modelo final con profundidad óptima ---
dt_model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
dt_model.fit(X_train, y_train)

print(f'Modelo final entrenado con max_depth={best_depth}.')
print(f'Número de nodos del árbol: {dt_model.tree_.node_count}')

## 4. Evaluación del Modelo

In [ ]:
y_pred = dt_model.predict(X_test)

r2   = r2_score(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)

print('=== Métricas de Evaluación en Datos de Test ===')
print(f'Coeficiente de Determinación (R²) : {r2:.4f}')
print(f'Error Cuadrático Medio (MSE)      : {mse:.4f}')
print(f'Raíz del Error Cuadrático Medio   : {rmse:.4f}')
print(f'Error Absoluto Medio (MAE)        : {mae:.4f}')

In [ ]:
# Gráfico: Valores Reales vs. Predichos
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='white', color='mediumslateblue', linewidths=0.5)
lims = [min(y_test.min(), y_pred.min()) - 0.5, max(y_test.max(), y_pred.max()) + 0.5]
plt.plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
plt.title(f'Valores Reales vs. Predichos (Árbol de Decisiones | R²={r2:.4f})')
plt.xlabel('Addicted Score Real')
plt.ylabel('Addicted Score Predicho')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Visualización del Árbol de Decisiones

In [ ]:
# Representación textual del árbol (primeras 4 ramas)
tree_rules = export_text(dt_model, feature_names=features, max_depth=4)
print(tree_rules)

In [ ]:
# Visualización gráfica del árbol (limitada a 3 niveles para mayor claridad)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_model,
    feature_names=features,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=10,
    ax=ax
)
plt.title('Árbol de Decisiones - Predicción de Adicción a Redes Sociales (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Importancia de Variables (Feature Importance)

In [ ]:
# Calcular y ordenar la importancia de las características
importances = pd.Series(dt_model.feature_importances_, index=features).sort_values(ascending=True)

colors = ['#6366f1' if imp == importances.max() else '#a5b4fc' for imp in importances]

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color=colors)
plt.title('Importancia de Variables en el Árbol de Decisiones')
plt.xlabel('Importancia Relativa (Gini Impurity Reduction)')
plt.tight_layout()
plt.show()

print('\nImportancia de cada variable:')
for feat, imp in importances.sort_values(ascending=False).items():
    print(f'  {feat:<32}: {imp:.4f} ({imp*100:.1f}%)')

## 7. Comparación: Regresión Lineal vs. Árbol de Decisiones

In [ ]:
# Cargar el modelo de Regresión Lineal para comparar
try:
    lr_model = joblib.load('modelo_adiccion.pkl')
    y_pred_lr = lr_model.predict(X_test)
    r2_lr    = r2_score(y_test, y_pred_lr)
    rmse_lr  = np.sqrt(mean_squared_error(y_test, y_pred_lr))
    mae_lr   = mean_absolute_error(y_test, y_pred_lr)
except FileNotFoundError:
    print('modelo_adiccion.pkl no encontrado. Ejecuta primero 03_Regresion_Lineal.ipynb.')
    r2_lr = rmse_lr = mae_lr = None

# Tabla comparativa
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Lineal', f'Árbol de Decisiones (depth={best_depth})'],
    'R²':   [r2_lr, r2],
    'RMSE': [rmse_lr, rmse],
    'MAE':  [mae_lr, mae]
})
print(comparacion.to_string(index=False))

# Gráfico de barras comparativo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
modelos = comparacion['Modelo']
paleta  = ['#6366f1', '#d946ef']

for ax, metric in zip(axes, ['R²', 'RMSE', 'MAE']):
    bars = ax.bar(modelos, comparacion[metric], color=paleta, width=0.5)
    ax.set_title(f'Comparación de {metric}', fontsize=13)
    ax.set_ylabel(metric)
    ax.set_xticklabels(modelos, rotation=12, ha='right', fontsize=10)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.suptitle('Regresión Lineal vs. Árbol de Decisiones', y=1.03, fontsize=14, fontweight='bold')
plt.show()

## 8. Exportación del Modelo

Guardamos el modelo de Árbol de Decisiones entrenado para su uso futuro.

In [ ]:
model_path = 'modelo_arbol.pkl'
joblib.dump(dt_model, model_path)
print(f'Modelo Árbol de Decisiones guardado exitosamente en "{model_path}".')

# Verificación de carga
loaded = joblib.load(model_path)
test_pred = loaded.predict([[20, 5.5, 6.0, 6, 2]])
print(f'Predicción de verificación: {test_pred[0]:.2f}')

## 9. Conclusiones

1. **Interpretabilidad**: El Árbol de Decisiones genera reglas explícitas y visualizables, lo que lo convierte en un modelo muy valioso para explicar las predicciones a stakeholders no técnicos.
2. **Variables más relevantes**: La visualización de importancias confirma que `Mental_Health_Score`, `Conflicts_Over_Social_Media` y `Avg_Daily_Usage_Hours` son los factores más determinantes del puntaje de adicción.
3. **Profundidad óptima**: La validación cruzada permite seleccionar la complejidad del árbol que maximiza la generalización, evitando el sobreajuste (overfitting) de un árbol sin poda.
4. **Comparación de modelos**: Ambos modelos son comparados mediante métricas estándar, permitiendo elegir el más adecuado según las necesidades del proyecto.